In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model_groq = init_chat_model("groq:openai/gpt-oss-120b")

## Pydantic


In [8]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    cast:str=Field(description="Cast of the movie")
    year:int=Field(description="Year of realse of movie")
    buget:int=Field(description="Buget of the movie in rupee")

In [3]:
movie_model = model_groq.with_structured_output(Movie) 

In [10]:
movie_model.invoke("tell me about movie Spiderman ")

Movie(title='Spider-Man', cast='Tobey Maguire, Kirsten Dunst, Willem Dafoe', year=2002, buget=139000000)

In [11]:
movie_model.invoke("tell me about movie Amazing Spiderman  ")

Movie(title='The Amazing Spider-Man', cast='Andrew Garfield, Emma Stone, Rhys Ifans, Sally Field, Martin Sheen', year=2012, buget=230000000)

### Message output alonside parsed structure

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(...,description="Title of the movie")
    cast:str=Field(...,description="Cast of the movie")
    year:int=Field(...,description="Year of realse of movie")
    buget:int=Field(...,description="Buget of the movie in rupee")

movie_raw_model = model_groq.with_structured_output(Movie,include_raw=True)


In [ ]:
movie_raw_model.invoke("tell me about movie Iron man  ")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me about movie Iron man". We need to provide information about the movie Iron Man. Could include budget, cast, title, year, plot summary, etc. There\'s a function available: Movie that takes budget, cast, title, year. We can call it to get structured info. Probably we should call the function with known data: Iron Man (2008), budget $140 million (in rupee? The function expects budget in rupee). We need to convert USD to INR. Approx exchange rate maybe 1 USD = 83 INR (as of 2024). So 140 million USD * 83 = 11,620 million INR, i.e., 11.62 billion INR. Could approximate 11.6 billion. Cast: Robert Downey Jr., Gwyneth Paltrow, Terrence Howard, Jeff Bridges, etc. Provide a summary.\n\nWe can call the function with these details.', 'tool_calls': [{'id': 'fc_6d8d39a2-6056-4119-a9d6-2b1ecc17c377', 'function': {'arguments': '{"buget":11620000000,"cast":"Robert Downey Jr., Gwyneth Paltrow, Terrence Howard,

### Nested Structure

In [18]:
class Actor(BaseModel):
    name:str
    role:str
    type:str

class Movie(BaseModel):
    title:str
    actor:list[Actor]
    genres:list[str]
    buget: float | None = Field(None)

movie_nested_model = model_groq.with_structured_output(Movie)



In [26]:
res = movie_nested_model.invoke("KGF")
res

Movie(title='KGF', actor=[Actor(name='Yash', role='Rocky', type='Lead'), Actor(name='Sanjay Dutt', role='Adheera', type='Antagonist'), Actor(name='Raveena Tandon', role='Ramika Sen', type='Supporting'), Actor(name='Ramachandra Raju', role='Garuda', type='Supporting')], genres=['Action', 'Drama'], buget=25000000.0)